# Rhythms of the Sleeping Brain
## SARIMA Modeling & Spectral Analysis of EEG Alpha Power Across Human Sleep Cycles

**Dataset:** PhysioNet Sleep-EDF Expanded Database (Kemp et al., 2000)  
**Recording:** `SC4001E0-PSG.edf` — Subject 1, overnight cassette PSG, channel `EEG Fpz-Cz`  
**Software:** Python 3.10 — `mne`, `scipy`, `statsmodels`, `numpy`, `pandas`, `matplotlib`

---

### What this notebook does

We take a raw overnight EEG recording and extract a single number per 30-second epoch: 
the total power in the **alpha band (8–13 Hz)**. This produces a univariate time series 
with ~2600 observations across ~22 hours. We then:

1. Assess stationarity and apply appropriate transformations  
2. Identify and fit a SARIMA model manually (no `auto_arima`)  
3. Validate the model through residual diagnostics  
4. Forecast the held-out 20% test window  
5. Perform spectral analysis including Fisher and KS tests  

### Why alpha power?

Alpha oscillations (8–13 Hz) are a reliable proxy for **arousal state**. They are suppressed 
during deep slow-wave sleep (N2/N3), elevated at sleep onset (N1), and transiently present 
during REM. A model that forecasts alpha power trajectory gives a closed-loop neural device 
a short look-ahead for sleep-state transitions — useful for adaptive stimulation thresholds.


## 1. Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import signal
from scipy.stats import probplot, shapiro, kstest
from scipy.special import comb
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
import warnings, os

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12})

# Paths and constants
EDF_FILE   = 'data/SC4001E0-PSG.edf'
HYPNO_FILE = 'data/SC4001EC-Hypnogram.edf'
EPOCH_SEC  = 30        # standard PSG epoch length (seconds)
ALPHA_LO   = 8.0       # alpha band lower bound (Hz)
ALPHA_HI   = 13.0      # alpha band upper bound (Hz)
FS         = 100       # target sampling rate (Hz)
TRAIN_FRAC = 0.80
S          = 18        # candidate seasonal period in epochs (18 × 30s = 9 min)

FIGURES_DIR = 'outputs/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs('outputs/models', exist_ok=True)
print('Setup complete.')

## 2. Data Loading

The recording is in **European Data Format (EDF)**, a standard for polysomnography. 
We use the `mne` library to read it. The channel `EEG Fpz-Cz` is a frontal midline 
derivation — one of the most informative sites for sleep-state alpha activity.

The recording is already at 100 Hz, so no resampling is needed.


In [ ]:
def load_edf_channel(edf_path, channel='EEG Fpz-Cz'):
    """
    Load a single EEG channel from an EDF file.
    Returns (signal_array, sampling_frequency).
    """
    import mne
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw.pick(channel)
    if raw.info['sfreq'] != FS:
        raw.resample(FS, npad='auto')
    data, _ = raw[:]
    return data.squeeze(), raw.info['sfreq']

raw_eeg, sfreq = load_edf_channel(EDF_FILE)
print(f'Signal shape : {raw_eeg.shape}')
print(f'Sampling rate: {sfreq} Hz')
print(f'Duration     : {len(raw_eeg)/sfreq/3600:.2f} hours')

## 3. Alpha Power Extraction

For each 30-second epoch we compute a single scalar: **alpha-band power**.

**Step 1 — Band-pass filter.** A 4th-order zero-phase Butterworth filter with 
passband [8, 13] Hz is applied per epoch using `sosfiltfilt` (second-order sections). 
Zero-phase filtering is essential on short segments: a standard causal filter introduces 
phase lag that distorts spectral estimates at short epoch lengths.

**Step 2 — Welch PSD.** We estimate the power spectral density using Welch's method 
with 2-second Hann-windowed segments and 50% overlap. This reduces the variance of the 
raw periodogram at the cost of some frequency resolution.

**Step 3 — Integrate.** Alpha power for epoch $t$ is:
$$X_t = \int_{8}^{13} \hat{P}_{xx}(f)\, df \approx \sum_{f_k \in [8,13]} \hat{P}_{xx}(f_k)\, \Delta f$$

This gives us a univariate time series of ~2650 values, one per 30-second epoch.


In [ ]:
def bandpass_filter(signal_arr, lo, hi, fs, order=4):
    """Zero-phase Butterworth band-pass via second-order sections."""
    nyq = 0.5 * fs
    sos = signal.butter(order, [lo/nyq, hi/nyq], btype='band', output='sos')
    return signal.sosfiltfilt(sos, signal_arr)


def compute_epoch_alpha_power(raw_signal, fs, epoch_sec=EPOCH_SEC,
                               lo=ALPHA_LO, hi=ALPHA_HI):
    """Construct the univariate alpha-power time series X_t."""
    N_ep = int(fs * epoch_sec)
    n_epochs = len(raw_signal) // N_ep
    alpha_power = np.empty(n_epochs)
    for t in range(n_epochs):
        seg = raw_signal[t*N_ep:(t+1)*N_ep]
        filt = bandpass_filter(seg, lo, hi, fs)
        freqs, pxx = signal.welch(filt, fs=fs, nperseg=int(fs*2),
                                   window='hann', scaling='density')
        mask = (freqs >= lo) & (freqs <= hi)
        alpha_power[t] = np.trapz(pxx[mask], freqs[mask])
    return alpha_power


print('Extracting alpha power per epoch (this takes ~30 seconds)...')
X_raw = compute_epoch_alpha_power(raw_eeg, sfreq)
X_t = pd.Series(X_raw, name='alpha_power_uV2')
print(f'Series length : {len(X_t)} epochs')
print(f'Duration      : {len(X_t)*EPOCH_SEC/3600:.1f} hours')
print(f'Value range   : [{X_t.min():.3e}, {X_t.max():.3e}] \u03bcV\u00b2/Hz')
print(f'Mean          : {X_t.mean():.3e}')
print(f'Skewness      : {X_t.skew():.3f}')

## 4. Train / Test Split

We hold out the last **20% of epochs** as a test set. This is a contiguous block 
at the end of the night — the correct approach for time series (never shuffle). 
All model identification and fitting happens on the training set only. The test 
set is touched exactly once, at validation.


In [ ]:
split_idx = int(len(X_t) * TRAIN_FRAC)
train = X_t.iloc[:split_idx].copy()
test  = X_t.iloc[split_idx:].copy()
print(f'Train: {len(train)} epochs ({len(train)*EPOCH_SEC/3600:.1f} hr)')
print(f'Test : {len(test)}  epochs ({len(test)*EPOCH_SEC/3600:.1f} hr)')

## 5. Exploratory Data Analysis

Before fitting anything, examine the raw series carefully. Look for:

- **Trend:** Does the mean drift up or down over time?
- **Seasonality:** Are there regular oscillating patterns?
- **Variance structure:** Does the spread (volatility) change with the level? 
  If high-power periods are also more volatile, this suggests a **multiplicative** 
  model — i.e., log transform is appropriate.
- **Sharp breaks:** Any sudden jumps that might require special handling?

The histogram helps assess skewness. Power spectral quantities are typically 
right-skewed (a few very high-power epochs pull the tail right).


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)

# Raw series
axes[0].plot(train.values, color='#4C72B0', lw=0.8)
axes[0].set_title('Training Series — Alpha Power (\u03bcV\u00b2/Hz)')
axes[0].set_xlabel('Epoch (30 s)'); axes[0].set_ylabel('Power')

# Rolling statistics (window = 6 epochs = 3 minutes)
roll = train.rolling(window=6)
axes[1].plot(train.values, alpha=0.35, color='#4C72B0', lw=0.7, label='Raw')
axes[1].plot(roll.mean().values, color='#C44E52', lw=1.5, label='Rolling mean (6 ep)')
axes[1].plot(roll.std().values,  color='#55A868', lw=1.5, label='Rolling std (6 ep)')
axes[1].set_title('Rolling Statistics — Trend & Variance Inspection')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=9)

# Marginal distribution
axes[2].hist(train.dropna(), bins=50, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[2].set_title('Marginal Distribution of Alpha Power')
axes[2].set_xlabel('Power (\u03bcV\u00b2/Hz)'); axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/eda_plots.png', bbox_inches='tight')
plt.show()

> **What to look for:** The rolling std panel is key. If the green line 
> (rolling std) tracks the red line (rolling mean) — rising and falling together — 
> that's evidence of multiplicative variance and justifies a log transform. 
> If the std is roughly flat regardless of the mean level, an additive model suffices.


## 6. Stationarity & Transformation

### 6.1 Log Transform

If the EDA showed variance proportional to level, we apply:
$$Y_t = \log(X_t + \varepsilon), \quad \varepsilon = 10^{-6} \cdot \text{median}(X_t)$$

The $\varepsilon$ shift prevents $\log(0)$ in rare zero-power epochs.

### 6.2 The Augmented Dickey-Fuller (ADF) Test

The ADF test checks for a **unit root in the autoregressive part** — i.e., stochastic 
trend. **Important limitation:** ADF does *not* detect other forms of non-stationarity 
such as seasonality, structural breaks, or heteroscedasticity. A series can pass ADF 
and still be non-stationary in other ways. We use it as one piece of evidence, not the 
whole picture.

- **H₀:** unit root present (non-stationary) 
- **Reject H₀** when ADF stat < critical value (equivalently p < 0.05)

We test at three stages: raw log series, after regular differencing, after seasonal differencing.


In [ ]:
def log_transform(series):
    """Y_t = log(X_t + eps). Variance-stabilising for multiplicative processes."""
    eps = 1e-6 * series.median()
    return np.log(series + eps)


def manual_difference(series, d=1, D=0, S=0):
    """
    Regular differencing (order d): removes polynomial trend
        nabla^d Y_t = (1-B)^d Y_t
    Seasonal differencing (order D, period S): removes period-S seasonality
        nabla_S^D Y_t = (1-B^S)^D Y_t
    Applied sequentially.
    """
    out = series.copy()
    for _ in range(d):
        out = out.diff(1).dropna()
    if D > 0 and S > 0:
        for _ in range(D):
            out = out.diff(S).dropna()
    return out


def run_adf_test(series, label=''):
    """
    ADF test with autolag='AIC'. Returns a result dict and prints a summary.
    Reminder: ADF only tests for unit root, not other non-stationarity types.
    """
    result = adfuller(series.dropna(), autolag='AIC')
    stat, pval = result[0], result[1]
    crits = result[4]
    print(f'ADF [{label}]')
    print(f'  Statistic : {stat:.4f}')
    print(f'  p-value   : {pval:.4f}')
    print(f'  Lags used : {result[2]}')
    print(f'  Crit 1%   : {crits["1%"]:.4f}  |  5%: {crits["5%"]:.4f}  |  10%: {crits["10%"]:.4f}')
    print(f'  Decision  : {"STATIONARY" if pval < 0.05 else "NON-STATIONARY"} at 5%')
    print()
    return {'label': label, 'stat': stat, 'pval': pval, 'crits': crits}

In [ ]:
eps_val = 1e-6 * X_t.median()

Y_t   = log_transform(train)          # log-transform of training set
W_t   = manual_difference(Y_t, d=1)   # one regular difference
W_t_s = manual_difference(Y_t, d=1, D=1, S=S)  # regular + seasonal difference

adf_log   = run_adf_test(Y_t,   label='log(X_t)')
adf_diff  = run_adf_test(W_t,   label='nabla log(X_t)')
adf_sdiff = run_adf_test(W_t_s, label=f'nabla nabla_{S} log(X_t)')

> **Reading the ADF results:** Our `log(X_t)` series already passes ADF (p ≈ 0), 
> which means no unit root is detected in the log series itself. However, ADF only 
> tests for stochastic trend — the visible oscillating structure in the EDA plots 
> (rolling mean is far from constant) suggests we still need differencing to achieve 
> practical stationarity for ARMA modeling. We proceed with d=1 and examine the 
> ACF/PACF at each stage to confirm.


## 7. ACF & PACF — Model Identification

The **ACF** (autocorrelation function) and **PACF** (partial autocorrelation function) 
are the primary tools for identifying ARIMA orders. The blue shaded region shows the 
95% confidence band: spikes outside it are statistically significant.

**Reading rules:**

| Pattern | Model implied |
|---|---|
| ACF cuts off at lag $q$, PACF tails off | MA($q$) |
| PACF cuts off at lag $p$, ACF tails off | AR($p$) |
| Both tail off gradually | Mixed ARMA($p$,$q$) |
| Significant spike at lag $S$ in ACF | Seasonal MA term |
| Significant spike at lag $S$ in PACF | Seasonal AR term |

We plot both the simply-differenced and seasonally-differenced series for comparison.


In [ ]:
def plot_acf_pacf(series, lags=60, title='', fname='acf_pacf.png'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(series.dropna(),  lags=lags, alpha=0.05, ax=ax1,
             title=f'ACF — {title}', zero=False)
    plot_pacf(series.dropna(), lags=lags, alpha=0.05, ax=ax2,
              title=f'PACF — {title}', zero=False, method='ywm')
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/{fname}', bbox_inches='tight')
    plt.show()

print('--- After regular differencing only ---')
plot_acf_pacf(W_t, lags=60, title='nabla log(X_t)', fname='acf_pacf_nodiff.png')

In [ ]:
print('--- After regular + seasonal differencing ---')
plot_acf_pacf(W_t_s, lags=60, title=f'nabla nabla_{S} log(X_t)', fname='acf_pacf.png')

> **Your job here:** Look at the ACF and PACF plots above and decide:
> - Does the ACF cut off or tail off at the non-seasonal lags (1–5)? → determines q
> - Does the PACF cut off or tail off at the non-seasonal lags (1–5)? → determines p
> - Is there a significant spike at lag 18? → determines whether SAR/SMA terms are needed
> - Does the seasonal differencing help or over-difference (variance increasing)?
>
> The candidate models below are based on the ACF/PACF patterns. They can be adjusted 
> after your own reading of the plots.


## 8. SARIMA Fitting & AICc Comparison

We fit a **SARIMA$(p,d,q)(P,D,Q)_S$** model:
$$\Phi_P(B^S)\,\phi_p(B)\,\nabla^d\nabla_S^D Y_t = \Theta_Q(B^S)\,\theta_q(B)\,\varepsilon_t, \quad \varepsilon_t \sim WN(0, \sigma^2)$$

where $B$ is the backshift operator, $Y_t = \log(X_t + \varepsilon)$.

The differencing is handled internally by `SARIMAX` — we pass the **log-transformed 
un-differenced** series and specify $d$ and $D$ in the order tuples.

**Model selection via AICc** (corrected AIC for small samples):
$$\text{AICc} = -2\ell(\hat{\theta}) + 2k + \frac{2k(k+1)}{n-k-1}$$

where $k$ = number of free parameters (including $\sigma^2$), $n$ = effective observations. 
Lower AICc = better model. A difference of >2 is meaningful; >10 is decisive.


In [ ]:
def compute_aicc(log_likelihood, n_params, n_obs):
    """AICc = AIC + 2k(k+1)/(n-k-1). Stronger small-sample penalty than AIC."""
    if n_obs - n_params - 1 <= 0:
        raise ValueError(f'Overparameterized: n={n_obs}, k={n_params}')
    aic  = -2.0 * log_likelihood + 2.0 * n_params
    corr = (2.0 * n_params * (n_params + 1)) / (n_obs - n_params - 1)
    return aic + corr


def fit_sarima(endog, order, seasonal_order, label):
    """Fit SARIMAX and report statistics manually. No automated wrappers."""
    model  = SARIMAX(endog, order=order, seasonal_order=seasonal_order,
                     enforce_stationarity=True, enforce_invertibility=True, trend='n')
    result = model.fit(disp=False, maxiter=200)
    n_obs    = result.nobs
    n_params = result.df_model + 1
    ll       = result.llf
    aicc_val = compute_aicc(ll, n_params, n_obs)
    print(f'{'='*55}')
    print(f'  {label}  SARIMA{order} x {seasonal_order}')
    print(f'  Log-likelihood : {ll:.4f}')
    print(f'  AIC  : {result.aic:.4f}   AICc : {aicc_val:.4f}   BIC : {result.bic:.4f}')
    print(f'  k={n_params}  n={n_obs}')
    print(f'  Coefficients:')
    for nm, val, se in zip(result.param_names, result.params, result.bse):
        print(f'    {nm:<22} = {val:+.6f}  (SE={se:.6f})')
    print(f'{'='*55}')
    return {'label': label, 'result': result, 'aicc': aicc_val,
            'order': order, 'seasonal_order': seasonal_order}

In [ ]:
# Candidate models — adjust orders based on your ACF/PACF reading above
model_A = fit_sarima(Y_t, order=(1,1,1), seasonal_order=(1,1,1,S), label='Model A')
print()
model_B = fit_sarima(Y_t, order=(2,1,1), seasonal_order=(0,1,1,S), label='Model B')

best = min([model_A, model_B], key=lambda m: m['aicc'])
print(f'\nBest by AICc: {best["label"]}  (AICc = {best["aicc"]:.4f})')

## 9. Residual Diagnostics

A well-fitted model should have residuals that look like **white noise** — 
uncorrelated, zero-mean, and ideally Gaussian. We check this four ways:

1. **Ljung-Box test on residuals** — tests for autocorrelation up to lag $m$  
   $H_0$: no autocorrelation. Want p > 0.05.

2. **Shapiro-Wilk test** — tests normality of residuals  
   $H_0$: residuals ~ Normal. Want p > 0.05 (often fails with EEG due to outliers).

3. **McLeod-Li test** — Ljung-Box applied to the **squared** residuals  
   Tests for nonlinear structure / ARCH effects.  
   $H_0$: no autocorrelation in $\varepsilon_t^2$. If rejected, the model misses 
   conditional heteroscedasticity (volatility clustering).

4. **Visual checks** — residual time plot, ACF of residuals, Q-Q plot.


In [ ]:
def mcleod_li_test(resid, lags=[10, 20, 30]):
    """McLeod-Li: Ljung-Box on squared residuals. Tests for ARCH/nonlinear effects."""
    lb = acorr_ljungbox(pd.Series(resid**2), lags=lags, return_df=True)
    print('McLeod-Li Test (Ljung-Box on squared residuals):')
    print(lb.to_string())
    return lb


def plot_residual_diagnostics(fit_result, label='', fname_prefix=''):
    resid = fit_result.resid.dropna().values
    fig = plt.figure(figsize=(14, 10))
    gs  = gridspec.GridSpec(2, 2, figure=fig)

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(resid, color='#4C72B0', lw=0.8)
    ax1.axhline(0, color='red', lw=0.8, ls='--')
    ax1.set_title(f'Residuals — {label}')
    ax1.set_xlabel('Epoch')

    ax2 = fig.add_subplot(gs[1, 0])
    plot_acf(resid, lags=40, alpha=0.05, ax=ax2, zero=False, title='ACF of Residuals')

    ax3 = fig.add_subplot(gs[1, 1])
    probplot(resid, dist='norm', plot=ax3)
    ax3.set_title('Q-Q Plot of Residuals')

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/residuals_{fname_prefix}.png', bbox_inches='tight')
    plt.show()

    # Formal tests
    print(f'\n--- Ljung-Box on residuals ({label}) ---')
    lb = acorr_ljungbox(resid, lags=[10, 20, 30], return_df=True)
    print(lb.to_string())

    stat, p_sw = shapiro(resid[:500])
    print(f'\nShapiro-Wilk (first 500): W={stat:.4f}  p={p_sw:.4f}')

    print()
    mcleod_li_test(resid, lags=[10, 20, 30])

In [ ]:
plot_residual_diagnostics(
    best['result'],
    label=best['label'],
    fname_prefix=best['label'].replace(' ', '_')
)

> **What the numbers mean:**  
> - Ljung-Box **p > 0.05 at all lags** → residuals are white noise ✓  
> - Ljung-Box **p < 0.05** at some lags → autocorrelation remains; model under-fitted  
> - McLeod-Li **p < 0.05** → squared residuals are correlated → possible ARCH effects  
> - Shapiro-Wilk **p < 0.05** → non-normal residuals (common with EEG; discuss but not fatal)  
> - ACF of residuals → all spikes should be within the blue ±1.96/√n bands  
> - Q-Q plot → should be a straight diagonal line for Gaussian residuals


## 10. Forecasting & Back-Transformation

The model produces forecasts on the **log scale**. To return to original units (μV²/Hz) 
we invert the log transform. A naive back-transform `exp(Ŷ)` would systematically 
underestimate the mean because exp of a Gaussian is log-normal:

$$E[e^Y] = e^{\mu + \sigma^2/2}$$

So the **bias-corrected** point forecast is:
$$\hat{X}_{n+h} = \exp\!\left(\hat{Y}_{n+h} + \frac{\hat{\sigma}^2_h}{2}\right) - \varepsilon$$

And the 95% prediction interval (direct inversion, no bias correction on bounds):
$$\left[\exp(\hat{Y}_{n+h} - 1.96\,\hat{\sigma}_h) - \varepsilon,\quad \exp(\hat{Y}_{n+h} + 1.96\,\hat{\sigma}_h) - \varepsilon\right]$$


In [ ]:
def forecast_and_backtransform(fit_dict, steps, eps):
    """Out-of-sample forecast on log scale, bias-corrected inversion to original scale."""
    result  = fit_dict['result']
    fc      = result.get_forecast(steps=steps)
    fc_mean = fc.predicted_mean
    fc_ci   = fc.conf_int(alpha=0.05)
    fc_var  = fc.var_pred_mean

    df = pd.DataFrame()
    df['log_mean'] = fc_mean.values
    df['log_lo']   = fc_ci.iloc[:, 0].values
    df['log_hi']   = fc_ci.iloc[:, 1].values
    df['fc_var']   = fc_var.values if hasattr(fc_var, 'values') else fc_var

    # Bias-corrected back-transform for point forecast
    df['forecast']  = np.exp(df['log_mean'] + df['fc_var'] / 2.0) - eps
    # Direct inversion for CI bounds
    df['lower_95']  = np.exp(df['log_lo']) - eps
    df['upper_95']  = np.exp(df['log_hi']) - eps
    return df


fc_df = forecast_and_backtransform(best, steps=len(test), eps=eps_val)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
n_train = len(train); n_test = len(test)
t_train = np.arange(n_train)
t_test  = np.arange(n_train, n_train + n_test)

ax.plot(t_train, train.values,         color='#4C72B0', lw=0.9, label='Training')
ax.plot(t_test,  test.values,          color='#55A868', lw=0.9, label='Actual (test)')
ax.plot(t_test,  fc_df['forecast'].values,
        color='#C44E52', lw=1.3, ls='--', label=f'Forecast ({best["label"]})')
ax.fill_between(t_test, fc_df['lower_95'].values, fc_df['upper_95'].values,
                color='#C44E52', alpha=0.15, label='95% CI')
ax.set_title(f'Alpha Power Forecast — {best["label"]}')
ax.set_xlabel('Epoch (30 s)'); ax.set_ylabel('Alpha Power (\u03bcV\u00b2/Hz)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/forecast.png', bbox_inches='tight')
plt.show()

# Accuracy metrics on original scale
actual = test.values
pred   = fc_df['forecast'].values[:len(actual)]
mae    = np.mean(np.abs(actual - pred))
rmse   = np.sqrt(np.mean((actual - pred)**2))
mape   = np.mean(np.abs((actual - pred) / (actual + 1e-30))) * 100
print(f'MAE  = {mae:.4e}')
print(f'RMSE = {rmse:.4e}')
print(f'MAPE = {mape:.2f}%')

## 11. Spectral Analysis (PSTAT 274)

Spectral analysis examines the **frequency content** of the alpha-power series. 
Where does the energy concentrate? Is there a dominant periodicity?

**Periodogram** (Schuster, non-parametric):
$$\hat{I}(\omega_j) = \frac{1}{n}\left|\sum_{t=1}^n X_t e^{-i\omega_j t}\right|^2$$

Frequencies are converted from cycles/epoch to **cycles/hour** for interpretability. 
Expected peaks:
- **~0.67 cycles/hr** (period ≈ 90 min) — ultradian NREM-REM cycle  
- **~6.67 cycles/hr** (period ≈ 9 min) — possible spindle-burst cadence

**Fisher's g-test:** Is the dominant peak statistically significant?
$$g = \frac{\max_j \hat{I}(\omega_j)}{\sum_j \hat{I}(\omega_j)}, \quad H_0: \text{white noise (flat spectrum)}$$
Exact p-value via the formula: $p = \sum_{k=1}^{\lfloor 1/g \rfloor} (-1)^{k-1}\binom{m}{k}(1-kg)^{m-1}$

**KS test on cumulative periodogram:** Under white noise, the normalized cumulative 
periodogram $C(k) = \sum_{j=1}^k \hat{I}_j / \sum_j \hat{I}_j$ should be Uniform(0,1). 
We apply a KS test to check this.


In [ ]:
def fisher_g_test(Pxx):
    """Fisher g-test: is the dominant periodogram peak significant?"""
    m   = len(Pxx)
    g   = np.max(Pxx) / np.sum(Pxx)
    upper = int(np.floor(1.0 / g))
    p_val = sum((-1)**(k-1) * float(comb(m, k, exact=True)) * (1 - k*g)**(m-1)
                for k in range(1, upper + 1))
    p_val = float(np.clip(p_val, 0.0, 1.0))
    print(f'Fisher g-test:  g = {g:.6f},  p = {p_val:.6f}')
    print(f'  -> {"Significant periodicity" if p_val < 0.05 else "No significant periodicity"} at 5%')
    return g, p_val


def ks_cumulative_periodogram(Pxx):
    """KS test: does the cumulative periodogram follow Uniform(0,1)?"""
    cumPxx = np.cumsum(Pxx) / np.sum(Pxx)
    ks_stat, p_val = kstest(cumPxx, 'uniform')
    print(f'KS cumulative periodogram:  D = {ks_stat:.6f},  p = {p_val:.6f}')
    print(f'  -> {"Spectral structure remains" if p_val < 0.05 else "Consistent with white noise"} at 5%')
    return ks_stat, p_val, cumPxx

In [ ]:
dt_hr = EPOCH_SEC / 3600.0  # hours per epoch

# Periodogram of full series X_t
f_ep, Pxx = signal.periodogram(X_t.values, fs=1.0)
f_hr      = f_ep / dt_hr
Pxx_nz    = Pxx[1:]    # exclude DC component
f_hr_nz   = f_hr[1:]

# Formal tests
g_stat, g_pval = fisher_g_test(Pxx_nz)
print()
ks_stat, ks_pval, cumPxx = ks_cumulative_periodogram(Pxx_nz)

# AR PSD (AIC-selected order)
best_aic = np.inf; best_ar = 1
for p in range(1, 26):
    try:
        m = AutoReg(X_t.values, lags=p, old_names=False).fit()
        if m.aic < best_aic:
            best_aic = m.aic; best_ar = p
    except Exception:
        pass
ar_fit    = AutoReg(X_t.values, lags=best_ar, old_names=False).fit()
ar_params = ar_fit.params[1:]
sigma2    = np.var(ar_fit.resid)
print(f'\nParametric AR order (AIC): {best_ar}')

w    = np.linspace(0, np.pi, 512)
H    = np.ones(len(w), dtype=complex)
for k, a_k in enumerate(ar_params, start=1):
    H -= a_k * np.exp(-1j * w * k)
S_ar = sigma2 / (np.abs(H)**2)
f_ar = (w / (2*np.pi)) / dt_hr

# Top peaks
print('\nTop-5 Periodogram Peaks:')
for idx in np.argsort(Pxx_nz)[-5:][::-1]:
    cyc_hr = f_hr_nz[idx]
    print(f'  {cyc_hr:.4f} cyc/hr  ==>  period ~ {60/cyc_hr:.1f} min  (Pxx={Pxx_nz[idx]:.3e})')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Periodogram
axes[0].semilogy(f_hr_nz, Pxx_nz, color='#4C72B0', lw=0.8)
axes[0].axvline(1/1.5,  color='red',    lw=1.3, ls='--', label='90-min NREM-REM (0.67 cyc/hr)')
axes[0].axvline(1/0.15, color='orange', lw=1.0, ls=':',  label='9-min spindle  (6.67 cyc/hr)')
axes[0].set_title('Periodogram of Alpha-Power Series $X_t$')
axes[0].set_xlabel('Frequency (cycles/hour)'); axes[0].set_ylabel('Power (log)')
axes[0].legend(fontsize=9)

# AR PSD
axes[1].semilogy(f_ar[1:], S_ar[1:], color='#C44E52', lw=1.0)
axes[1].axvline(1/1.5, color='navy', lw=1.3, ls='--', label=f'90-min cycle | AR({best_ar}) PSD')
axes[1].set_title(f'Parametric AR({best_ar}) Spectral Estimate')
axes[1].set_xlabel('Frequency (cycles/hour)'); axes[1].set_ylabel('PSD')
axes[1].legend(fontsize=9)

# Cumulative periodogram
freq_pos = np.linspace(0, 1, len(Pxx_nz))
axes[2].plot(freq_pos, cumPxx, color='#4C72B0', lw=1.2, label='Cumulative periodogram')
axes[2].plot(freq_pos, freq_pos, color='red', lw=1.0, ls='--', label='Expected (white noise)')
axes[2].set_title(f'Cumulative Periodogram  (KS D={ks_stat:.4f}, p={ks_pval:.4f})')
axes[2].set_xlabel('Normalized frequency'); axes[2].set_ylabel('Cumulative spectral mass')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/spectral_analysis.png', bbox_inches='tight')
plt.show()

## 12. Results Summary & Next Steps

Run this cell after completing the full analysis to print a consolidated summary 
of all key numbers for the report.


In [ ]:
print('=== RESULTS SUMMARY ===')
print(f'Series: {len(X_t)} epochs  ({len(X_t)*EPOCH_SEC/3600:.1f} hr)')
print(f'Train : {len(train)}  Test: {len(test)}')
print()
print('--- ADF Tests ---')
for r in [adf_log, adf_diff, adf_sdiff]:
    print(f"  {r['label']:<30}  stat={r['stat']:7.4f}  p={r['pval']:.4f}  "
          f"{'STATIONARY' if r['pval']<0.05 else 'NON-STATIONARY'}")
print()
print('--- Model Comparison ---')
for m in [model_A, model_B]:
    marker = ' <-- BEST' if m['label'] == best['label'] else ''
    print(f"  {m['label']}: SARIMA{m['order']}x{m['seasonal_order']}  AICc={m['aicc']:.4f}{marker}")
print()
print('--- Spectral ---')
print(f'  Fisher g={g_stat:.6f}  p={g_pval:.6f}')
print(f'  KS     D={ks_stat:.6f}  p={ks_pval:.6f}')
print()
print(f'--- Forecast Accuracy ({best["label"]}) ---')
print(f'  MAPE = {mape:.2f}%')
print(f'  RMSE = {rmse:.4e}  MAE = {mae:.4e}')